# 05 · gold/processing ➜ gold

This is the **Silver-to-Gold notebook**. Silver holds clean but *separate* tables (flow, temperature, humidity, weight, weather),
each with its own rhythm: flow every **minute**, hive temperature every **5 minutes**, humidity and weight every **12 hours**, weather every **hour**.
Gold puts them together into **tables that are ready to analyse**: one question, one table.

### Where this notebook sits in the pipeline
```
gold/processing/<place>_<sensor>__<run>.parquet     (handed over by notebook 04)
        │
        │  Data Factory: Notebook activity → THIS NOTEBOOK
        ▼
gold/hivedata_hourly__<run>.parquet          ★ main table: bee traffic + weather + hive sensors, one row per hour
gold/hivedata_daily__<run>.parquet           bonus: daily totals and averages
gold/activity_by_time_of_day__<run>.parquet  bonus: when are bees most active?
gold/activity_by_temperature__<run>.parquet  bonus: how does outside temperature change activity?
        │
        │  on success: Data Factory deletes gold/processing/*
```

### Why one row per hour?
If we merged everything **on the exact timestamp**, the sensors' different rhythms would leave most cells empty:
1.26 million rows, with weather filled in only 1 row in 60. The LMS asks to
*"ensure timestamps are properly aligned so that each record corresponds to the correct weather conditions at that moment"*.
Weather is measured **hourly**, so we bring everything to **hours**:
- bee counts are **added up** per hour (60 one-minute counts → bees per hour),
- hive temperature is **averaged** per hour,
- humidity and weight (12-hourly) appear in the hour they were measured,
- weather already is hourly.

Result: about 21,000 rows per hive, every row with weather. That's small, fast and easy to analyse.
The daily table **sums** counts and **averages** measurements, so each column keeps its meaning
(averaging bee counts would turn *bees per day* into *average bees per minute*).

## 0. Settings: the only cell you change when moving between your laptop and Azure
The notebook **detects by itself** where it runs:
- on your **laptop** it uses the folders of the **Bee Haven- Azure** project (the notebook sits in `notebooks/`),
- in **Synapse** it reads and writes the Azure Data Lake (`abfss://...`). Fill in your container, storage account and key below; the notebook stops with a clear message if you forget.

**First time on your laptop?** These notebooks need `fsspec` (Synapse already has it). Run once in a cell: `%pip install fsspec requests pyarrow`, then restart.

**Never paste a real account key into a notebook you share or upload.** Anyone who sees the key controls the whole data lake.
Better options, from best to OK:
1. In Synapse, read the key from **Azure Key Vault** (commented line below).
2. Put it in the notebook only while testing, and delete it before sharing.

In [ ]:
import re
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import fsspec

# Detect automatically where we run: Synapse has the 'notebookutils' package, a laptop doesn't.
try:
    from notebookutils import mssparkutils
    RUN_LOCAL = False                # running in Synapse / Azure
except ImportError:
    RUN_LOCAL = True                 # running on the laptop

if RUN_LOCAL:
    root_dir = Path('..').resolve().as_posix() + '/'          # the "Bee Haven- Azure" folder
    storage_options = {}
else:
    # abfss://<container>@<storage account>.dfs.core.windows.net/
    root_dir = 'abfss://<YOUR CONTAINER>@<YOUR STORAGE ACCOUNT>.dfs.core.windows.net/'
    storage_options = {
        'account_name': '<YOUR STORAGE ACCOUNT>',
        'account_key': '<YOUR ACCOUNT KEY>',   # Storage account -> Security + networking -> Access keys
        # safer: 'account_key': mssparkutils.credentials.getSecret('<key vault name>', '<secret name>'),
    }
    if '<YOUR' in root_dir or '<YOUR' in str(storage_options):
        raise ValueError('Fill in your container, storage account and key in this cell first.')

# the medallion folders
bronze_new             = root_dir + 'bronze/new/'
bronze_archive_weather = root_dir + 'bronze/archive/weather/'
silver_processing_path = root_dir + 'silver/processing/'
silver_path            = root_dir + 'silver/'
gold_processing        = root_dir + 'gold/processing/'
gold_path              = root_dir + 'gold/'

PLACES = {                                       # hive locations we know about
    'schwartau': {'lat': 53.919444, 'lon': 10.6975},
    'wurzburg':  {'lat': 49.783333, 'lon': 9.933333},
}
SENSORS = ['flow', 'humidity', 'temperature', 'weight']
LOCAL_TZ = 'Europe/Berlin'                       # hive loggers write German clock time

RUN_STAMP = pd.Timestamp.now(tz='UTC').strftime('%Y-%m-%dT%Hh%Mm%Ss')   # one stamp for every file this run writes
print('root:', root_dir)
print('run stamp:', RUN_STAMP)

## 1. Small file helpers
**Why:** reading and writing in Azure needs `fsspec` (the "Azure file system" library), while your laptop uses normal folders.
These helpers hide the difference, so the rest of the notebook is *exactly the same* in both places.
`fsspec` picks the right file system from the path itself (`abfss://...` → Azure, `/Users/...` → local disk).

In [ ]:
FS, _ = fsspec.core.url_to_fs(root_dir, **storage_options)

def list_files(folder, pattern='*'):
    """Files in a folder matching a pattern, ignoring hidden files like .DS_Store."""
    return sorted(p for p in FS.glob(folder + pattern) if not p.rsplit('/', 1)[-1].startswith('.'))

def file_name(path):
    return path.rstrip('/').rsplit('/', 1)[-1]

def _make_folder(path):
    folder = path.rsplit('/', 1)[0]
    try:
        FS.makedirs(folder, exist_ok=True)
    except Exception:
        pass                          # Azure Data Lake creates folders automatically when writing

def read_csv(path, **kwargs):
    with FS.open(path, 'rb') as f:
        return pd.read_csv(f, **kwargs)

def read_parquet(path):
    with FS.open(path, 'rb') as f:
        return pd.read_parquet(f)

def write_parquet(df, path):
    _make_folder(path)
    with FS.open(path, 'wb') as f:
        df.to_parquet(f, index=False)
    print(f'  wrote {path}  ({len(df):,} rows)')

def write_json(data, path):
    _make_folder(path)
    with FS.open(path, 'w') as f:
        json.dump(data, f, indent=2)

def read_json(path):
    with FS.open(path, 'r') as f:
        return json.load(f)

def exists(path):
    return FS.exists(path)

## 2. Read everything waiting in gold/processing
File names look like `schwartau_flow__2026-10-08T17h40m00s.parquet` → place `schwartau`, table `flow`.

**New data drops:** if the pipeline ran more than once before this notebook ran (e.g. the "New Data Drop" lesson), there can be
**several files of the same table**. Merging `flow` with `flow` would create broken columns like `flow_out_x`, `flow_out_y`,
so files of the same table are **stacked** (concatenated) instead, and if two files contain the same timestamp the **newest file wins**.

In [ ]:
TABLES = SENSORS + ['weather']
pieces = {}                         # pieces[place][table] = list of DataFrames (oldest file first)

for path in list_files(gold_processing, '*.parquet'):
    name = file_name(path)
    head = name.split('__')[0]                     # 'schwartau_flow'
    place, _, table = head.partition('_')
    if place not in PLACES or table not in TABLES:
        print(f'SKIPPED {name}: unknown place or table'); continue
    pieces.setdefault(place, {}).setdefault(table, []).append(read_parquet(path))
    print(f'read {name}')

if not pieces:
    raise FileNotFoundError(f'Nothing in {gold_processing}. Did notebook 04 run?')

silver = {}                         # silver[place][table] = one DataFrame
for place, tables in pieces.items():
    silver[place] = {}
    for table, frames in tables.items():
        df = pd.concat(frames, ignore_index=True)
        df = df.drop_duplicates(subset='timestamp', keep='last').sort_values('timestamp').reset_index(drop=True)
        silver[place][table] = df
        print(f'{place:10s} {table:12s} {len(frames)} file(s) → {len(df):>10,} rows')

## 3. Build the main gold table: one row per hour
**How each column is made**

| Gold column | From | How | Why |
|---|---|---|---|
| `departures`, `arrivals` | flow (per minute) | **sum** per hour; departures turned positive | "Bees per hour" is the natural unit; positive counts are easier to read than negative ones |
| `flow_minutes` | flow | number of minutes with data in that hour | Tells the analyst if an hour is complete (60) or partial |
| `temperature_hive` | hive temperature (5 min) | mean per hour | Renamed, so it can't be confused with the outside temperature |
| `humidity_hive`, `weight` | 12-hourly | value of the hour it was measured | Not spread into other hours, since we don't invent data |
| weather columns | weather (hourly) | as is | Already hourly |
| `location` | file name | category | So several hives can live in one table |

**Columns dropped** (LMS: *drop unnecessary columns*): the weather `*_source_distance` columns. They remain in silver for anyone who needs them.
`min_count=1` makes an hour with **no** data stay empty (NaN) instead of looking like "0 bees".

In [ ]:
WEATHER_COLS = ['temperature', 'relative_humidity', 'precipitation', 'sunshine', 'cloud_cover', 'wind_speed',
                'wind_gust_speed', 'wind_direction', 'wind_gust_direction', 'pressure_msl', 'dew_point', 'solar', 'condition']

def hourly_table(place, t):
    parts = []
    if 'flow' in t:
        f = t['flow'].set_index('timestamp')
        parts.append(pd.DataFrame({
            'departures': f['flow_out'].astype('float64').abs().resample('60min').sum(min_count=1),
            'arrivals':   f['flow_in'].astype('float64').resample('60min').sum(min_count=1),
            'flow_minutes': f['flow_out'].resample('60min').count(),
        }))
    if 'temperature' in t:
        parts.append(t['temperature'].set_index('timestamp')['temperature'].resample('60min').mean().rename('temperature_hive'))
    if 'humidity' in t:
        parts.append(t['humidity'].set_index('timestamp')['humidity'].resample('60min').mean().rename('humidity_hive'))
    if 'weight' in t:
        parts.append(t['weight'].set_index('timestamp')['weight'].resample('60min').mean())
    if 'weather' in t:
        w = t['weather'].set_index('timestamp')
        w.index = w.index.floor('60min')
        w = w[~w.index.duplicated(keep='last')]
        parts.append(w[[c for c in WEATHER_COLS if c in w.columns]])

    hourly = pd.concat(parts, axis=1, sort=True)
    hourly.index.name = 'timestamp'
    hourly = hourly.reset_index()
    hourly.insert(1, 'location', place)
    return hourly

COLUMN_ORDER = ['timestamp', 'location', 'departures', 'arrivals', 'flow_minutes',
                'temperature_hive', 'humidity_hive', 'weight'] + WEATHER_COLS

hourly = pd.concat([hourly_table(p, t) for p, t in silver.items()], ignore_index=True)
hourly = hourly[[c for c in COLUMN_ORDER if c in hourly.columns]]
hourly['location'] = hourly['location'].astype('category')
if 'flow_minutes' in hourly:
    hourly['flow_minutes'] = hourly['flow_minutes'].fillna(0).astype('int64')
print(hourly.shape)
hourly.head()

### Quality checks on the gold table
One row per hour per location, UTC timestamps, no negative counts, and weather present in (almost) every hour that has bee traffic.

In [ ]:
assert str(hourly['timestamp'].dt.tz) == 'UTC'
assert not hourly.duplicated(subset=['location', 'timestamp']).any(), 'duplicate hours'
assert (hourly['departures'].dropna() >= 0).all() and (hourly['arrivals'].dropna() >= 0).all()
assert hourly['flow_minutes'].between(0, 60).all()

with_flow = hourly['departures'].notna()
coverage = hourly.loc[with_flow, 'temperature'].notna().mean() * 100
print(f'hours with bee traffic that also have outside temperature: {coverage:.1f} %')
assert coverage > 95, 'weather is missing for many hours. Check the weather step in notebook 04'
hourly.info()

## 4. Bonus table: daily aggregates
**Sum** what is counted (bees, rain, sunshine minutes), **average** what is measured (temperatures, humidity, weight).
Plus `weight_change`: the change in hive weight compared with the previous day. In notebook 01 we saw that weight is a *running total*,
so the **change** per day is what relates to bee activity (honey coming in, or being eaten).

In [ ]:
def daily_table(df):
    d = df.set_index('timestamp')
    out = pd.DataFrame({
        'departures':         d['departures'].resample('D').sum(min_count=1),
        'arrivals':           d['arrivals'].resample('D').sum(min_count=1),
        'flow_minutes':       d['flow_minutes'].resample('D').sum(),
        'temperature_hive':   d['temperature_hive'].resample('D').mean(),
        'humidity_hive':      d['humidity_hive'].resample('D').mean(),
        'weight':             d['weight'].resample('D').mean(),
        'temperature_mean':   d['temperature'].resample('D').mean(),
        'temperature_min':    d['temperature'].resample('D').min(),
        'temperature_max':    d['temperature'].resample('D').max(),
        'precipitation':      d['precipitation'].resample('D').sum(min_count=1),
        'sunshine_hours':     d['sunshine'].resample('D').sum(min_count=1) / 60,
        'relative_humidity':  d['relative_humidity'].resample('D').mean(),
        'wind_speed':         d['wind_speed'].resample('D').mean(),
        'cloud_cover':        d['cloud_cover'].resample('D').mean(),
    })
    out['weight_change'] = out['weight'].diff()
    return out

daily = pd.concat({loc: daily_table(g) for loc, g in hourly.groupby('location', observed=True)}, names=['location', 'date'])
daily = daily.reset_index()
daily['location'] = daily['location'].astype('category')
daily['date'] = daily['date'].dt.date
print(daily.shape)
daily.describe().round(1)

## 5. Bonus tables: time of day, and temperature bands
**Time of day:** buckets use **German local time** (bees follow the sun, not UTC).
Morning 06–11, afternoon 12–17, evening 18–21, night 22–05. The values are average bees **per hour**.

**Temperature bands:** average bees per hour for each 5 °C band of outside temperature, counting **daylight hours only (08–18 local)**.
Otherwise all the zero-traffic night hours would drag every band down.

In [ ]:
def time_of_day(hour):
    if 6 <= hour <= 11:  return 'morning'
    if 12 <= hour <= 17: return 'afternoon'
    if 18 <= hour <= 21: return 'evening'
    return 'night'

act = hourly[hourly['departures'].notna()].copy()
act['local_hour'] = act['timestamp'].dt.tz_convert(LOCAL_TZ).dt.hour
act['time_of_day'] = pd.Categorical(act['local_hour'].map(time_of_day),
                                    categories=['morning', 'afternoon', 'evening', 'night'], ordered=True)

by_time_of_day = (act.groupby(['location', 'time_of_day'], observed=True)
                  .agg(hours=('departures', 'size'),
                       departures_per_hour=('departures', 'mean'),
                       arrivals_per_hour=('arrivals', 'mean'),
                       temperature_mean=('temperature', 'mean'))
                  .round(1).reset_index())

day = act[act['local_hour'].between(8, 18)].copy()
bins = [-np.inf, 0, 5, 10, 15, 20, 25, 30, np.inf]
labels = ['below 0', '0-5', '5-10', '10-15', '15-20', '20-25', '25-30', '30+']
day['temperature_band'] = pd.cut(day['temperature'], bins=bins, labels=labels, right=False)
by_temperature = (day.groupby(['location', 'temperature_band'], observed=True)
                  .agg(hours=('departures', 'size'),
                       departures_per_hour=('departures', 'mean'),
                       arrivals_per_hour=('arrivals', 'mean'),
                       sunshine_minutes=('sunshine', 'mean'))
                  .round(1).reset_index())

display(by_time_of_day)
display(by_temperature)

## 6. Write the gold tables
Same run stamp for all four files, so they always belong together.

In [ ]:
write_parquet(hourly,         f'{gold_path}hivedata_hourly__{RUN_STAMP}.parquet')
write_parquet(daily,          f'{gold_path}hivedata_daily__{RUN_STAMP}.parquet')
write_parquet(by_time_of_day, f'{gold_path}activity_by_time_of_day__{RUN_STAMP}.parquet')
write_parquet(by_temperature, f'{gold_path}activity_by_temperature__{RUN_STAMP}.parquet')

## 7. Quick visual check
**Why:** one look at a chart catches problems that numbers hide (a month missing, a unit wrong).
In the pipeline (`RUN_LOCAL = False`) charts are skipped to save Spark time. Set `SHOW_CHARTS = True` to see them in Synapse.

In [ ]:
SHOW_CHARTS = RUN_LOCAL

if SHOW_CHARTS:
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    BLUE, ORANGE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'
    plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.25,
                         'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'legend.frameon': False})

    for loc, d in daily.groupby('location', observed=True):
        d = d.assign(date=pd.to_datetime(d['date']))
        fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
        axes[0].plot(d['date'], d['departures'], color=ORANGE, label='departures')
        axes[0].plot(d['date'], d['arrivals'], color=AQUA, label='arrivals')
        axes[0].set_ylabel('bees per day'); axes[0].legend(loc='upper left')
        axes[0].set_title(f'{loc}: daily bee traffic (line chart, change over time)')
        axes[1].plot(d['date'], d['temperature_mean'], color=BLUE)
        axes[1].set_ylabel('°C'); axes[1].set_title('Outside temperature, daily mean')
        axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        plt.tight_layout(); plt.show()

    for loc, b in by_temperature.groupby('location', observed=True):
        fig, ax = plt.subplots(figsize=(10, 3.8))
        ax.bar(b['temperature_band'].astype(str), b['departures_per_hour'], color=BLUE, width=0.6)
        ax.set_xlabel('outside temperature band (°C)'); ax.set_ylabel('departures per hour')
        ax.set_title(f'{loc}: daytime activity by outside temperature (bar chart, comparing categories)')
        ax.grid(axis='x', visible=False)
        plt.show()

## 8. (Local only) Empty gold/processing, as Data Factory does on success
In Azure: a **Delete activity** on the success arrow of this notebook's activity, pointing at `gold/processing/`.

In [ ]:
if RUN_LOCAL:
    for path in list_files(gold_processing, '*.parquet'):
        FS.rm(path)
    print('gold/processing emptied')

## Release the Spark pool (Synapse only)
**Why:** in Synapse you pay for the Spark pool while the session is open. Stopping it as soon as the work is done saves your Azure credits.

In [ ]:
if not RUN_LOCAL:
    mssparkutils.session.stop()